# gl-yolo — sweep complet (MVA → DOTA → VisDrone)

Entraîne les 8 architectures sur les 3 datasets (24 runs), 200 epochs, imgsz=1280.

**Avant de lancer** : Runtime → Change runtime type → GPU (A100 si Colab Pro+ le propose).

**Ce run va prendre plusieurs jours.** Le repo est cloné *dans Drive* (pas sur le disque local éphémère) pour que `runs/` et `logs/` survivent aux déconnexions de session : à chaque reconnexion, ré-exécutez les cellules dans l'ordre — `run_gpu.py` saute automatiquement les modèles déjà terminés et reprend le modèle interrompu depuis son dernier checkpoint au lieu de le refaire depuis zéro.

## 1. Vérifier le GPU attribué

In [ ]:
!nvidia-smi
import torch
print("CUDA disponible:", torch.cuda.is_available())

## 2. Monter Google Drive (persistance entre sessions)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Cloner (1re fois) ou mettre à jour le repo sur Drive

In [ ]:
import os

REPO_DIR = "/content/drive/MyDrive/gl-yolo"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/osiastossou/gl-yolo.git "{REPO_DIR}"
else:
    print("Repo déjà présent sur Drive, mise à jour...")
    !git -C "{REPO_DIR}" pull

%cd {REPO_DIR}

## 4. Installer les dépendances

In [ ]:
!pip install -q -e .

## 5. Datasets téléchargés en local, pas sur Drive

Important pour la vitesse : `runs/`/`logs/` doivent persister sur Drive (survit aux déconnexions), mais les datasets sont statiques -- les lire en entraînement depuis Drive monté est lent (beaucoup de petits fichiers). On les télécharge/dézippe sur le disque local éphémère de Colab à chaque session (quelques minutes), bien plus rapide pendant tout le reste de l'entraînement.

In [ ]:
from ultralytics import settings

settings.update({"datasets_dir": "/content/datasets"})
print(settings["datasets_dir"])

## 6. (Optionnel) vérifier l'état d'un sweep précédent

Utile après une reconnexion, pour voir ce qui est déjà fini avant de relancer.

In [ ]:
!find runs -name best.pt 2>/dev/null
!find runs -name last.pt 2>/dev/null

## 7. Lancer le sweep complet (en arrière-plan)

MVA (bird, single-class) en premier, puis DOTA, puis VisDrone — 8 architectures chacun.

Lancé en arrière-plan (`subprocess.Popen`) exprès : ça libère immédiatement le kernel Colab, pour pouvoir exécuter la cellule de suivi (section 8) **pendant** que le sweep tourne, sans attendre qu'il se termine. Activez aussi l'**exécution en arrière-plan** de Colab (icône dans la barre d'outils, fonctionnalité Pro+) pour ne pas perdre le run en fermant l'onglet.

In [ ]:
import subprocess

sweep_log = open("sweep_colab.log", "w")
sweep_proc = subprocess.Popen(
    ["python", "run_gpu.py", "-epochs", "200", "-imgsz", "1280"],
    stdout=sweep_log,
    stderr=subprocess.STDOUT,
)
print("Sweep lancé en arrière-plan, PID:", sweep_proc.pid)

## 8. Suivre la progression

Ré-exécutez cette cellule à tout moment (elle ne bloque pas, pas besoin d'arrêter le sweep) pour voir : quel job tourne, l'epoch en cours et le détail du batch courant, et les métriques déjà sauvegardées pour ce job.

In [ ]:
import glob
import os

print("--- Sweep : dernier job démarré/terminé ---")
!tail -5 sweep_colab.log 2>/dev/null

log_files = glob.glob("logs/*.log")
if not log_files:
    print("\nPas encore de log -- le premier job n'a pas encore démarré.")
else:
    latest = max(log_files, key=os.path.getmtime)
    name = os.path.splitext(os.path.basename(latest))[0]
    print(f"\n--- Job en cours : {name} (epoch/batch courant) ---")
    # le log contient des retours chariot (\r) pour la barre de progression -> on les
    # convertit en vrais saut de lignes pour voir le dernier état proprement
    !tail -c 4000 "{latest}" | tr '\r' '\n' | tail -5

    csv_path = f"runs/{name}/results.csv"
    if os.path.exists(csv_path):
        import pandas as pd

        print(f"\n--- Epochs déjà sauvegardés ({csv_path}) ---")
        print(pd.read_csv(csv_path).tail(5).to_string(index=False))
    else:
        print("\nPas encore de checkpoint sauvegardé pour ce job (results.csv pas encore créé).")

## En cas de déconnexion

Reconnectez-vous, ré-exécutez les cellules **2 à 7** dans l'ordre (Drive → repo → deps → datasets_dir → sweep). `run_gpu.py` saute les modèles déjà terminés (`best.pt` présent) et reprend automatiquement celui qui était en cours (`last.pt` sans `best.pt`) depuis son dernier checkpoint.